# Logistic Regression Experiments

This notebook documents all logistic regression experiments on the fraud detection dataset.
Each experiment is logged as a separate section with its own cells, showing the progression
of improvements and the reasoning behind each change.

**Dataset:** `data/processed/selected_features.csv` (20 MRMR-selected features + target)

**Split strategy:** 70% train / 15% validation / 15% test (stratified)

**Why stratified split?** The dataset is imbalanced (~75% non-fraud, ~25% fraud). Stratification
ensures each split preserves this ratio, preventing a split where one set has disproportionate
class distribution which would give misleading metrics.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score
)
import warnings
warnings.filterwarnings('ignore')

from src.data.load_data import load_selected_features
from src.data.preprocess import encode_target, encode_categoricals, scale_features, split_data
from src.models.predict import predict, predict_proba
from src.evaluation.metrics import evaluate_model, print_metrics

print('Imports loaded successfully.')

## Data Preparation

We load the MRMR-selected features from the feature selection step. The preprocessing pipeline:
1. **Label encoding** for categorical features — converts string categories to integers so logistic regression can process them
2. **Stratified train/val/test split** — 70/15/15 ratio, preserving class balance in each set
3. **Standard scaling** — fit on training set only, then transform val/test to prevent data leakage

**Why scale?** Logistic regression uses gradient-based optimization (L-BFGS). Features on different scales
(e.g., `total_claim_amount` in thousands vs `witnesses` in 0-5) cause the optimizer to take inefficient
zigzag paths. Scaling ensures all features contribute proportionally and regularization (C parameter)
penalizes coefficients fairly.

In [ ]:
# Load data
df = load_selected_features('../data/processed/selected_features.csv')
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['fraud_reported'].value_counts()}")
print(f"Fraud rate: {(df['fraud_reported'] == 'Y').mean()*100:.1f}%")

# Separate features and target
X = df.drop(columns=['fraud_reported'])
y = encode_target(df['fraud_reported'])

# Encode categoricals
X_encoded, encoders = encode_categoricals(X)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")

In [ ]:
# Split data: 70% train, 15% validation, 15% test
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X_encoded, y, test_size=0.15, val_size=0.15, random_state=42
)

print(f"Train set:      {X_train.shape[0]} samples ({y_train.mean()*100:.1f}% fraud)")
print(f"Validation set: {X_val.shape[0]} samples ({y_val.mean()*100:.1f}% fraud)")
print(f"Test set:       {X_test.shape[0]} samples ({y_test.mean()*100:.1f}% fraud)")

# Scale features (fit on train only)
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)
print(f"\nScaling: fit on train, transformed all sets")

---
## Run 1: Baseline Logistic Regression

**Goal:** Establish a baseline with default logistic regression parameters.

**Configuration:**
- `C=1.0` — default regularization strength (inverse of lambda). C=1 is a moderate regularization.
- `class_weight='balanced'` — automatically adjusts weights inversely proportional to class frequencies.
  Without this, the model would optimize for the majority class (non-fraud) and predict almost everything
  as non-fraud, achieving ~75% accuracy but missing most fraud cases.
- `solver='lbfgs'` — Limited-memory Broyden-Fletcher-Goldfarb-Shanno, a quasi-Newton method good for
  small-to-medium datasets. Efficient and handles L2 regularization natively.
- `max_iter=1000` — ensures convergence (default 100 can sometimes be insufficient).

**Why balanced class weights?** In fraud detection, a false negative (missing a fraud case) is typically
more costly than a false positive (flagging a legitimate claim). Balanced weights tell the model that
misclassifying a fraud case is ~3x worse than misclassifying a non-fraud case (proportional to 750/250).

In [ ]:
# Run 1: Baseline
model_v1 = LogisticRegression(
    C=1.0,
    l1_ratio=0,          # L2 regularization (equivalent to old penalty='l2')
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
model_v1.fit(X_train_s, y_train)

# Evaluate on all three sets
results_log = []  # track all runs

for name, X_set, y_set in [("Train", X_train_s, y_train), ("Validation", X_val_s, y_val), ("Test", X_test_s, y_test)]:
    y_pred = predict(model_v1, X_set)
    y_prob = predict_proba(model_v1, X_set)
    res = evaluate_model(y_set, y_pred, y_prob)
    print_metrics(res, name)
    print()
    if name == "Validation":
        val_res_v1 = res
    if name == "Test":
        test_res_v1 = res

results_log.append({
    'Run': 'V1: Baseline (C=1.0, balanced)',
    'Val F1': val_res_v1['f1'],
    'Val Precision': val_res_v1['precision'],
    'Val Recall': val_res_v1['recall'],
    'Val ROC AUC': val_res_v1['roc_auc'],
    'Test F1': test_res_v1['f1'],
    'Test ROC AUC': test_res_v1['roc_auc'],
})

In [ ]:
# Confusion matrix for baseline
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, X_set, y_set) in zip(axes, [("Validation", X_val_s, y_val), ("Test", X_test_s, y_test)]):
    y_pred = predict(model_v1, X_set)
    cm = confusion_matrix(y_set, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
    ax.set_title(f'V1 Baseline - {name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

---
## Run 2: Cross-Validation to Assess Stability

**Goal:** Before tuning hyperparameters, assess how stable the baseline model is across different
data folds. This tells us if the validation score is reliable or just lucky.

**Why cross-validation here?** With only 700 training samples, a single train/val split can be noisy.
5-fold CV gives us 5 different validation estimates, and the variance tells us how sensitive the
model is to which data ends up in train vs validation.

**Method:** 5-fold stratified CV on the training set. We use stratified folds to maintain class
balance in each fold, and score with F1 since accuracy is misleading on imbalanced data.

**Note:** We run CV on training data only. The test set remains untouched until final evaluation.

In [ ]:
# Run 2: Cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_model = LogisticRegression(
    C=1.0, l1_ratio=0, solver='lbfgs', max_iter=1000,
    class_weight='balanced', random_state=42
)

# Score with multiple metrics
for metric in ['f1', 'precision', 'recall', 'roc_auc']:
    scores = cross_val_score(cv_model, X_train_s, y_train, cv=cv, scoring=metric)
    print(f"{metric:<12}: {scores.mean():.4f} (+/- {scores.std():.4f})  | folds: {[f'{s:.3f}' for s in scores]}")

print("\nInterpretation:")
print("- Low std means the model is stable across different data splits")
print("- High std would suggest the model is sensitive to which samples are in the training set")

---
## Run 3: Regularization Tuning (C parameter)

**Goal:** Find the optimal regularization strength.

**Why tune C?** The C parameter controls the trade-off between fitting the training data and keeping
coefficients small (generalization):
- **Low C** (e.g., 0.01) = strong regularization = simpler model, may underfit
- **High C** (e.g., 100) = weak regularization = complex model, may overfit
- We search across a log-scale range to find the sweet spot

**Method:** Evaluate C values on the validation set. We use the validation set (not CV here) because
we want a quick comparison across many C values. The best C will later be validated with CV.

In [ ]:
# Run 3: C parameter search
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
c_results = []

for C in C_values:
    model_c = LogisticRegression(
        C=C, l1_ratio=0, solver='lbfgs', max_iter=1000,
        class_weight='balanced', random_state=42
    )
    model_c.fit(X_train_s, y_train)
    
    y_pred_val = predict(model_c, X_val_s)
    y_prob_val = predict_proba(model_c, X_val_s)
    val_res = evaluate_model(y_val, y_pred_val, y_prob_val)
    
    y_pred_train = predict(model_c, X_train_s)
    train_f1 = f1_score(y_train, y_pred_train)
    
    c_results.append({
        'C': C,
        'Train F1': train_f1,
        'Val F1': val_res['f1'],
        'Val Precision': val_res['precision'],
        'Val Recall': val_res['recall'],
        'Val ROC AUC': val_res['roc_auc'],
    })

c_df = pd.DataFrame(c_results)
print(c_df.to_string(index=False))

best_c = c_df.loc[c_df['Val F1'].idxmax(), 'C']
print(f"\nBest C by validation F1: {best_c}")

In [ ]:
# Visualize C tuning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 scores
axes[0].semilogx(c_df['C'], c_df['Train F1'], 'o-', label='Train F1')
axes[0].semilogx(c_df['C'], c_df['Val F1'], 's-', label='Val F1')
axes[0].axvline(x=best_c, color='red', linestyle='--', alpha=0.5, label=f'Best C={best_c}')
axes[0].set_xlabel('C (regularization)')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score vs Regularization Strength')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision vs Recall trade-off across C
axes[1].semilogx(c_df['C'], c_df['Val Precision'], 'o-', label='Precision')
axes[1].semilogx(c_df['C'], c_df['Val Recall'], 's-', label='Recall')
axes[1].axvline(x=best_c, color='red', linestyle='--', alpha=0.5, label=f'Best C={best_c}')
axes[1].set_xlabel('C (regularization)')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision/Recall vs Regularization Strength')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Large gap between Train and Val F1 = overfitting. Look for C where the gap is small.")

---
## Run 4: Best C with Cross-Validation Confirmation

**Goal:** Confirm the best C found in Run 3 is genuinely better, not just lucky on this particular
validation split.

**Method:** Run 5-fold stratified CV with the best C on the full training+validation data to get
a robust performance estimate. Then retrain on full train set and evaluate on validation.

In [ ]:
# Run 4: Best C with CV confirmation
model_v2 = LogisticRegression(
    C=best_c, l1_ratio=0, solver='lbfgs', max_iter=1000,
    class_weight='balanced', random_state=42
)

# CV on training data
cv_scores = cross_val_score(model_v2, X_train_s, y_train, cv=cv, scoring='f1')
print(f"CV F1 with C={best_c}: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"CV F1 with C=1.0 (baseline): was computed in Run 2")

# Train and evaluate
model_v2.fit(X_train_s, y_train)

for name, X_set, y_set in [("Train", X_train_s, y_train), ("Validation", X_val_s, y_val), ("Test", X_test_s, y_test)]:
    y_pred = predict(model_v2, X_set)
    y_prob = predict_proba(model_v2, X_set)
    res = evaluate_model(y_set, y_pred, y_prob)
    print_metrics(res, name)
    print()
    if name == "Validation":
        val_res_v2 = res
    if name == "Test":
        test_res_v2 = res

results_log.append({
    'Run': f'V2: Tuned C={best_c}',
    'Val F1': val_res_v2['f1'],
    'Val Precision': val_res_v2['precision'],
    'Val Recall': val_res_v2['recall'],
    'Val ROC AUC': val_res_v2['roc_auc'],
    'Test F1': test_res_v2['f1'],
    'Test ROC AUC': test_res_v2['roc_auc'],
})

---
## Run 5: Threshold Tuning

**Goal:** The default classification threshold is 0.5, but this may not be optimal for imbalanced data.

**Why tune the threshold?** Logistic regression outputs a probability. The threshold decides where we
draw the line between "fraud" and "not fraud":
- **Lower threshold** (e.g., 0.3) = more fraud predictions = higher recall, lower precision
- **Higher threshold** (e.g., 0.7) = fewer fraud predictions = lower recall, higher precision

In fraud detection, we typically prefer higher recall (catch more fraud) even at the cost of some
precision (more false alarms), because the cost of missing fraud >> cost of investigating a false alarm.

**Method:** Sweep thresholds from 0.1 to 0.9 on the validation set and pick the one that maximizes F1.

In [ ]:
# Run 5: Threshold tuning on validation set
y_prob_val = predict_proba(model_v2, X_val_s)

thresholds = np.arange(0.1, 0.91, 0.05)
threshold_results = []

for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t):
        continue  # skip trivial predictions
    res = evaluate_model(y_val, y_pred_t, y_prob_val)
    threshold_results.append({
        'Threshold': t,
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1': res['f1'],
        'Accuracy': res['accuracy'],
    })

t_df = pd.DataFrame(threshold_results)
print(t_df.to_string(index=False))

best_threshold = t_df.loc[t_df['F1'].idxmax(), 'Threshold']
print(f"\nBest threshold by F1: {best_threshold:.2f}")

In [ ]:
# Visualize threshold trade-off
plt.figure(figsize=(10, 5))
plt.plot(t_df['Threshold'], t_df['Precision'], 'o-', label='Precision')
plt.plot(t_df['Threshold'], t_df['Recall'], 's-', label='Recall')
plt.plot(t_df['Threshold'], t_df['F1'], '^-', label='F1', linewidth=2)
plt.axvline(x=best_threshold, color='red', linestyle='--', alpha=0.5, label=f'Best threshold={best_threshold:.2f}')
plt.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
plt.xlabel('Classification Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Classification Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate with best threshold on all sets
print(f"Using threshold = {best_threshold:.2f} (instead of default 0.5)\n")

for name, X_set, y_set in [("Train", X_train_s, y_train), ("Validation", X_val_s, y_val), ("Test", X_test_s, y_test)]:
    y_prob = predict_proba(model_v2, X_set)
    y_pred_t = (y_prob >= best_threshold).astype(int)
    res = evaluate_model(y_set, y_pred_t, y_prob)
    print_metrics(res, name)
    print()
    if name == "Validation":
        val_res_v3 = res
    if name == "Test":
        test_res_v3 = res

results_log.append({
    'Run': f'V3: Tuned C={best_c}, threshold={best_threshold:.2f}',
    'Val F1': val_res_v3['f1'],
    'Val Precision': val_res_v3['precision'],
    'Val Recall': val_res_v3['recall'],
    'Val ROC AUC': val_res_v3['roc_auc'],
    'Test F1': test_res_v3['f1'],
    'Test ROC AUC': test_res_v3['roc_auc'],
})

---
## Run 6: No Class Weighting (Comparison)

**Goal:** Understand the effect of `class_weight='balanced'` by removing it.

**Why this experiment?** We need to quantify how much the balanced weighting helps. Without it,
the model treats all misclassifications equally, which on imbalanced data typically leads to:
- Higher accuracy (predicts majority class more)
- Lower recall (misses more fraud cases)
- This comparison justifies our choice of balanced weights.

In [ ]:
# Run 6: Without class_weight='balanced' for comparison
model_noweight = LogisticRegression(
    C=best_c, l1_ratio=0, solver='lbfgs', max_iter=1000,
    class_weight=None,  # no class weighting
    random_state=42
)
model_noweight.fit(X_train_s, y_train)

print("WITHOUT class_weight='balanced':")
for name, X_set, y_set in [("Validation", X_val_s, y_val), ("Test", X_test_s, y_test)]:
    y_pred = predict(model_noweight, X_set)
    y_prob = predict_proba(model_noweight, X_set)
    res = evaluate_model(y_set, y_pred, y_prob)
    print_metrics(res, name)
    print()
    if name == "Validation":
        val_res_noweight = res
    if name == "Test":
        test_res_noweight = res

results_log.append({
    'Run': f'V4: No class weighting (C={best_c})',
    'Val F1': val_res_noweight['f1'],
    'Val Precision': val_res_noweight['precision'],
    'Val Recall': val_res_noweight['recall'],
    'Val ROC AUC': val_res_noweight['roc_auc'],
    'Test F1': test_res_noweight['f1'],
    'Test ROC AUC': test_res_noweight['roc_auc'],
})

print("Comparison: balanced vs no weighting shows the impact on recall (fraud detection rate)")

---
## Run Summary & Final Evaluation

Compare all experiment runs side-by-side, then do the final evaluation on the test set with
the best configuration. The test set has been held out throughout all tuning — this is its
first and only use for model selection.

In [ ]:
# Summary table of all runs
summary_df = pd.DataFrame(results_log)
print("=" * 100)
print("EXPERIMENT LOG - All Runs")
print("=" * 100)
print(summary_df.to_string(index=False))

# Identify best run by validation F1
best_idx = summary_df['Val F1'].idxmax()
print(f"\nBest run by Val F1: {summary_df.loc[best_idx, 'Run']}")
print(f"  Val F1:      {summary_df.loc[best_idx, 'Val F1']:.4f}")
print(f"  Test F1:     {summary_df.loc[best_idx, 'Test F1']:.4f}")
print(f"  Test ROC AUC: {summary_df.loc[best_idx, 'Test ROC AUC']:.4f}")

In [ ]:
# Final model: ROC Curve and Precision-Recall Curve on test set
y_prob_test = predict_proba(model_v2, X_test_s)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
roc_auc_val = auc(fpr, tpr)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (Test Set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob_test)
axes[1].plot(recall_curve, precision_curve, 'r-', linewidth=2, label='Precision-Recall')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Test Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance: logistic regression coefficients
# The magnitude and sign of coefficients tell us which features push toward fraud vs non-fraud
coef_df = pd.DataFrame({
    'Feature': X_train_s.columns,
    'Coefficient': model_v2.coef_[0]
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
plt.barh(range(len(coef_df)), coef_df['Coefficient'], color=colors)
plt.yticks(range(len(coef_df)), coef_df['Feature'])
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression Coefficients (Green = pushes toward Fraud)')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Positive coefficient = higher value increases fraud probability")
print("Negative coefficient = higher value decreases fraud probability")

In [ ]:
# Final test set classification report
y_pred_final = predict(model_v2, X_test_s)
print("Final Test Set Classification Report")
print("=" * 50)
print(classification_report(y_test, y_pred_final, target_names=['Non-Fraud', 'Fraud']))